<small>Dataset: Iris (built into scikit-learn)</small>

# Q18. Iris KNN

Use nearby labeled examples to classify new records, and compare three K values.


## Question from Classroom

Iris Flower Classification Using KNN 
Problem statement: Classify Iris flowers into their species using the K-Nearest Neighbors 
algorithm. 
Tasks to be performed: 
1. Load the Iris dataset. 
2. Separate the four measurement features and the species label. 
3. Split the data into training and testing sets. 
4. Standardize the features using training-set statistics. 
5. Train KNN classifiers using at least three different K values. 
6. Compare test accuracy and select a suitable K.


## Understand before coding

Choose X/y -> stratified split -> training-only imputation and scaling -> compare K=3,5,7 -> print metrics and unseen predictions.


Run the code cells from top to bottom with **Shift + Enter**. If an import is missing, run the following installation line once in a new cell, then restart the kernel. The line below is a comment until you remove `#`.


**Optional package setup (run once in a new Jupyter cell if needed):**

```python
%pip install pandas numpy matplotlib scikit-learn openpyxl
```


**Code layout:** 6 listed tasks → 6 code blocks. Run them in order. Imports and helper definitions are included in the relevant task. Where the PDF repeats a task number, these blocks follow its listed order. Learned preprocessing is fitted only after splitting.


## Task 1. Load the Iris dataset.

This block implements the task above. Variables from earlier blocks are reused.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

from sklearn.datasets import load_iris
iris=load_iris(as_frame=True)
X,y=iris.data,iris.target
print("Features:",X.columns.tolist(),"Species:",iris.target_names)


Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)'] Species: ['setosa' 'versicolor' 'virginica']


## Task 2. Separate the four measurement features and the species label.

This block implements the task above. Variables from earlier blocks are reused.


In [2]:
X=iris.data
y=iris.target
print("Four feature measurements:");print(X.head())
print("Species labels:",iris.target_names)


Four feature measurements:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2
3                4.6               3.1                1.5               0.2
4                5.0               3.6                1.4               0.2
Species labels: ['setosa' 'versicolor' 'virginica']


## Task 3. Split the data into training and testing sets.

This block implements the task above. Variables from earlier blocks are reused.


In [3]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


## Task 4. Standardize the features using training-set statistics.

This block implements the task above. Variables from earlier blocks are reused.


In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

scaler=StandardScaler()
train_scaled=scaler.fit_transform(X_train)
test_scaled=scaler.transform(X_test)
print("Scaler fitted on training rows only. CV below refits its own scaler inside each fold.")


Scaler fitted on training rows only. CV below refits its own scaler inside each fold.


## Task 5. Train KNN classifiers using at least three different K values.

This block implements the task above. Variables from earlier blocks are reused.


In [5]:
models={}
results=[]
for k in [3,5,7]:
    model=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),KNeighborsClassifier(n_neighbors=k))
    cv_accuracy=cross_val_score(model,X_train,y_train,cv=3,scoring="accuracy").mean()
    model.fit(X_train,y_train)
    models[k]=model
    results.append({"K":k,"Training CV accuracy":cv_accuracy,"Test accuracy":accuracy_score(y_test,model.predict(X_test))})
table=pd.DataFrame(results)


## Task 6. Compare test accuracy and select a suitable K.

This block implements the task above. Variables from earlier blocks are reused.


In [6]:
print(table.to_string(index=False))
best_k=int(table.loc[table["Training CV accuracy"].idxmax(),"K"])
print("K selected using training CV:",best_k)
model=models[best_k]
prediction=model.predict(X_test)

print("Accuracy:",round(accuracy_score(y_test,prediction),4))
print("Class order:",model.classes_)
print("Confusion matrix:\n",confusion_matrix(y_test,prediction,labels=model.classes_))
print(classification_report(y_test,prediction,zero_division=0))


 K  Training CV accuracy  Test accuracy
 3              0.958333       0.933333
 5              0.958333       0.933333
 7              0.950000       0.966667
K selected using training CV: 3
Accuracy: 0.9333
Class order: [0 1 2]
Confusion matrix:
 [[10  0  0]
 [ 0 10  0]
 [ 0  2  8]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.83      1.00      0.91        10
           2       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30



## Viva preparation guide



**1. What are X and y?**

X contains the predictor columns; y contains the target we want to predict. X is a 2D table and y is a 1D series.

**2. Why split training and testing data?**

Fit using the training examples. Evaluate on separate examples to estimate how well the model generalizes.

**3. What do fit() and predict() mean?**

fit() learns parameters from training examples. predict() applies the learned model to new feature rows.

**4. Why random_state=42?**

It makes random splits or model choices reproducible. The number 42 has no special mathematical meaning.

**5. What is data leakage?**

Information unavailable for a real prediction accidentally influences training. Examples: including the answer in X or fitting a scaler on test data. Fit learned preprocessing only on training data.

**6. What is overfitting?**

The model learns training-specific details/noise and does worse on unseen data. Control complexity and use validation or cross-validation.

**7. What is a hyperparameter?**

A setting chosen outside model fitting, such as max_depth, K or the number of trees. Choose it using training validation, not by repeatedly optimizing the final test result.

**8. How do I fix FileNotFoundError or KeyError?**

Open Jupyter in the praceice folder, keep data/ beside this notebook, and print df.columns.tolist() to check exact column names. Do not change X and y row counts separately.

**9. What does the confusion matrix contain?**

Rows are actual classes and columns are predicted classes, in the printed label order. For binary labels [0,1], it is [[TN,FP],[FN,TP]].

**10. Define accuracy, precision, recall and F1.**

Accuracy=(TP+TN)/N. Precision=TP/(TP+FP). Recall=TP/(TP+FN). F1=2PR/(P+R). classification_report prints per-class values and macro/weighted averages. zero_division=0 handles an undefined score.

**11. Why can accuracy be misleading?**

If most examples belong to one class, always predicting that class may give high accuracy. Inspect minority-class recall, precision, F1 and the confusion matrix.

**12. Why stratify=y?**

It approximately preserves class proportions in training and test splits. Each class needs enough examples.

**13. How does KNN classify a record?**

Measure distance to training examples, take the K nearest, and predict by their class votes.

**14. Why scale features?**

A large numeric range could dominate distance. StandardScaler subtracts the training mean and divides by training standard deviation.

**15. Why try K=3,5,7?**

The question requests multiple K values. They illustrate the bias/variance tradeoff; three candidates do not guarantee the best possible K.

**16. Why choose K by training CV?**

Repeatedly choosing the best final test score would make the test set part of tuning. We show the requested test comparisons but select K using training cross-validation.

**17. What if K is very small or large?**

Small K can follow noise and overfit. Very large K can smooth away useful local patterns and underfit.

**18. Does an odd K prevent every tie?**

No. It helps avoid a two-class even-vote tie, but multiclass votes can still tie.

**19. Is KNN eager or lazy?**

It is called lazy learning: it retains training examples and performs much of its work during prediction.

**20. What does the pipeline do?**

Impute training medians, fit a training scaler, then fit KNN. During CV each fold fits its own preprocessing, preventing leakage.

**21. Complexity?**

Brute-force prediction needs roughly O(n*p) distance work per query for n training rows and p features, plus neighbour selection. Storage is roughly O(n*p); indexed algorithms vary.

**22. What are Iris features and labels?**

Sepal length, sepal width, petal length and petal width predict setosa, versicolor or virginica. Built-in targets are 0,1,2 in the printed species order.


### Task-to-code map

- **Block 1:** Load the Iris dataset.
- **Block 2:** Separate the four measurement features and the species label.
- **Block 3:** Split the data into training and testing sets.
- **Block 4:** Standardize the features using training-set statistics.
- **Block 5:** Train KNN classifiers using at least three different K values.
- **Block 6:** Compare test accuracy and select a suitable K.
